In [ ]:
# Dataset manipulation
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import category_encoders as ce
import utils_pipeline_dhm as up

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import StratifiedKFold

# Metrics
from sklearn.metrics import (recall_score, 
                             classification_report,
                             roc_auc_score,
                             average_precision_score
                            )

# Others
import importlib

In [23]:
# read data
data_path = '../data/diabetes_prediction_dataset.csv'
data = pd.read_csv(data_path, sep=',')
print('Filas del dataset en bruto:', data.shape[0])
# Filter data
data = data[data['bmi'] < 65]
data = data[data['gender']!='Other']
print('Filas del dataset después de filtrar:', data.shape[0])
# Remap 'smoking_history' values
smoking_mapping = {
    'No Info': 'No Info',
    'never': 'never',
    'former': 'former',
    'current': 'current',
    'not current': 'former',
    'ever': 'former'
}

data['smoking_history'] = data['smoking_history'].map(smoking_mapping)

# check data
data.head()

Filas del dataset en bruto: 100000
Filas del dataset después de filtrar: 99935


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [25]:
# Features and target
X = data.drop(columns=["diabetes"])
y = data["diabetes"]   # binary: 0/1

# Split first (important to avoid leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [47]:
# Declare columns
binary_cols = ['hypertension','heart_disease']
woe_cat_cols = ['gender','smoking_history','hypertension','heart_disease']

# Numerical columns to bin and their max number of bins
binning_config = {
    "blood_glucose_level": 4,
    'HbA1c_level': 5,
    "age": 8,
    "bmi": 8
}

# Transformations
binary_to_cat = up.TypeCaster(
    columns=binary_cols,
    dtype=str
)

woe_transformer = Pipeline([
    ("woe", ce.WOEEncoder()),
    ("invert", FunctionTransformer(lambda X: -X,feature_names_out="one-to-one"))
])

# Preprocessor
preprocessor = ColumnTransformer([
    ("woe", woe_transformer, woe_cat_cols),
    ("bin_woe",up.MultiOptimalBinningWOE(binning_config),list(binning_config.keys()))
    ],
remainder="passthrough",
verbose_feature_names_out=True
)

# Pipeline
log_pipe_line = Pipeline([
    ("type_cast", binary_to_cat),
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

# Scoring metrics
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

# Hyperparameters for Grid Search
param_grid = {
    "model__C": [0.01, 0.1]
}

# Cross-validation strategy with K-Folds and shuffling
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid search with cross-validation
log_grid = GridSearchCV(
    log_pipe_line,
    param_grid=param_grid,
    cv=cv,
    scoring=scoring,
    refit="pr_auc",
    n_jobs=-1
)

# Fit the model
log_grid.fit(X_train, y_train)

# Extract and display CV results
cv_results = pd.DataFrame(log_grid.cv_results_)
cv_results = cv_results[["param_model__C", 
                         "mean_test_roc_auc",
                         "mean_test_pr_auc",
                         'std_test_roc_auc', 
                         'std_test_pr_auc']]

print(cv_results,'\n')

# best model
best_params = log_grid.best_params_
best_score = log_grid.best_score_

print("Stats for best parameters:")
print("Best Parameters:", best_params)
print("Mean PR-AUC Score:", best_score,'\n')

# Get the best model and evaluate on both Train and Test sets
best_model = log_grid.best_estimator_
threshold = 0.5  # Set threshold for classification

# Print Train metrics for the best model
print("Train Metrics for the Best Model:")
train_preds = best_model.predict_proba(X_train)[:,1]
print('Train PR-AUC',average_precision_score(y_train, train_preds))
print('Train ROC-AUC',roc_auc_score(y_train, train_preds))
print('Train Recall:',recall_score(y_train, (train_preds >= threshold).astype(int)),'\n')

# print Test metrics for the best model
print("Test Metrics for the Best Model:")
test_preds = best_model.predict_proba(X_test)[:,1]
print('Test PR-AUC',average_precision_score(y_test, test_preds))
print('Test ROC-AUC',roc_auc_score(y_test, test_preds))
print('Test Recall:',recall_score(y_test, (test_preds >= threshold).astype(int)))

   param_model__C  mean_test_roc_auc  mean_test_pr_auc  std_test_roc_auc  \
0            0.01            0.93643          0.679025          0.002299   
1            0.10            0.93648          0.679350          0.002246   

   std_test_pr_auc  
0         0.004332  
1         0.004141   

Stats for best parameters:
Best Parameters: {'model__C': 0.1}
Mean PR-AUC Score: 0.6793498411985077 

Train Metrics for the Best Model:
Train PR-AUC 0.6825054263489353
Train ROC-AUC 0.9374574686442129
Train Recall: 0.4963170300530348 

Test Metrics for the Best Model:
Test PR-AUC 0.6811171131739508
Test ROC-AUC 0.9406898547045199
Test Recall: 0.4938126104890984


In [50]:
from sklearn.feature_selection import RFECV

# Modified pipeline without the model
preprocessor = ColumnTransformer([
    ("woe", woe_transformer, woe_cat_cols),
    ("bin_woe", up.MultiOptimalBinningWOE(binning_config), list(binning_config.keys()))
], remainder="passthrough", verbose_feature_names_out=True)

# Pipeline with RFE
log_pipe_line = Pipeline([
    ("type_cast", binary_to_cat),
    ("preprocessor", preprocessor),
    ("rfe", RFECV(
        estimator=LogisticRegression(max_iter=1000, random_state=42),
        step=1,
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring="roc_auc",
        n_jobs=-1
    )),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Rest of your GridSearchCV remains the same
param_grid = {
    "model__C": [0.01, 0.1]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_grid = GridSearchCV(
    log_pipe_line,
    param_grid=param_grid,
    cv=cv,
    scoring=scoring,
    refit="pr_auc",
    n_jobs=-1
)

log_grid.fit(X_train, y_train)

# Extract and display CV results
cv_results = pd.DataFrame(log_grid.cv_results_)
cv_results = cv_results[["param_model__C", 
                         "mean_test_roc_auc",
                         "mean_test_pr_auc",
                         'std_test_roc_auc', 
                         'std_test_pr_auc']]

print(cv_results,'\n')

# best model
best_params = log_grid.best_params_
best_score = log_grid.best_score_

print("Stats for best parameters:")
print("Best Parameters:", best_params)
print("Mean PR-AUC Score:", best_score,'\n')

# Get the best model and evaluate on both Train and Test sets
best_model = log_grid.best_estimator_
threshold = 0.5  # Set threshold for classification

# Print Train metrics for the best model
print("Train Metrics for the Best Model:")
train_preds = best_model.predict_proba(X_train)[:,1]
print('Train PR-AUC',average_precision_score(y_train, train_preds))
print('Train ROC-AUC',roc_auc_score(y_train, train_preds))
print('Train Recall:',recall_score(y_train, (train_preds >= threshold).astype(int)),'\n')

# print Test metrics for the best model
print("Test Metrics for the Best Model:")
test_preds = best_model.predict_proba(X_test)[:,1]
print('Test PR-AUC',average_precision_score(y_test, test_preds))
print('Test ROC-AUC',roc_auc_score(y_test, test_preds))
print('Test Recall:',recall_score(y_test, (test_preds >= threshold).astype(int)))

# View selected features
best_model = log_grid.best_estimator_
selected_features = best_model.named_steps['rfe'].get_feature_names_out()
print(f"Selected features ({len(selected_features)}):\n{selected_features}")

   param_model__C  mean_test_roc_auc  mean_test_pr_auc  std_test_roc_auc  \
0            0.01            0.93643          0.679025          0.002299   
1            0.10            0.93648          0.679350          0.002246   

   std_test_pr_auc  
0         0.004332  
1         0.004141   

Stats for best parameters:
Best Parameters: {'model__C': 0.1}
Mean PR-AUC Score: 0.6793498411985077 

Train Metrics for the Best Model:
Train PR-AUC 0.6825054263489353
Train ROC-AUC 0.9374574686442129
Train Recall: 0.4963170300530348 

Test Metrics for the Best Model:
Test PR-AUC 0.6811171131739508
Test ROC-AUC 0.9406898547045199
Test Recall: 0.4938126104890984
Selected features (8):
['x0' 'x1' 'x2' 'x3' 'x4' 'x5' 'x6' 'x7']


In [53]:
# STEP 1: GridSearch to find best C parameter
print("="*60)
print("STEP 1: Finding Best Hyperparameters with GridSearch")
print("="*60)

log_pipe_line = Pipeline([
    ("type_cast", binary_to_cat),
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_grid = GridSearchCV(
    log_pipe_line,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",  # Use single metric for hyperparameter tuning
    n_jobs=-1
)

log_grid.fit(X_train, y_train)

# Extract best parameters
best_C = log_grid.best_params_['model__C']
best_score = log_grid.best_score_

print(f"\n✓ Best C: {best_C}")
print(f"✓ Best CV ROC-AUC: {best_score:.4f}\n")

# ============================================================
# STEP 2: Use BEST parameters for Backward Elimination
# ============================================================
print("="*60)
print("STEP 2: Backward Elimination with Best Parameters")
print("="*60)

def backward_elimination_optimized(X, y, best_C, cv, metric='roc_auc'):
    """
    Backward elimination using the best hyperparameters found in GridSearch
    """
    current_features = list(X.columns)
    results = []
    
    while len(current_features) > 1:
        # Create a fresh preprocessor for CURRENT features
        current_woe_cols = [col for col in woe_cat_cols if col in current_features]
        current_bin_cols = [col for col in binning_config.keys() if col in current_features]
        
        current_preprocessor = ColumnTransformer([
            ("woe", woe_transformer, current_woe_cols),
            ("bin_woe", up.MultiOptimalBinningWOE({k: v for k, v in binning_config.items() if k in current_bin_cols}), current_bin_cols)
        ], remainder="passthrough", verbose_feature_names_out=True)
        
        # Train model with BEST C parameter and CURRENT features
        pipe = Pipeline([
            ("type_cast", binary_to_cat),
            ("preprocessor", current_preprocessor),
            ("model", LogisticRegression(C=best_C, max_iter=1000, random_state=42))
        ])
        
        scores = cross_validate(
            pipe, X[current_features], y, 
            cv=cv, scoring=metric, return_train_score=True
        )
        
        mean_score = scores['test_score'].mean()
        std_score = scores['test_score'].std()
        
        results.append({
            'iteration': len(results),
            'n_features': len(current_features),
            'cv_score': mean_score,
            'cv_std': std_score,
            'feature_list': current_features.copy()
        })
        
        print(f"Iteration {len(results):2d} | Features: {len(current_features):2d} | "
              f"ROC-AUC: {mean_score:.4f} ± {std_score:.4f}")
        
        # Train on full training set to get coefficients
        pipe.fit(X[current_features], y)
        coefficients = pipe.named_steps['model'].coef_[0]
        
        # Find feature with smallest absolute coefficient
        min_idx = np.argmin(np.abs(coefficients))
        removed_feature = current_features[min_idx]
        removed_coef = coefficients[min_idx]
        
        print(f"           → Removing: '{removed_feature}' (coef: {removed_coef:.4f})\n")
        current_features.pop(min_idx)
    
    results_df = pd.DataFrame(results)
    
    # Find best iteration (highest CV score)
    best_idx = results_df['cv_score'].idxmax()
    best_features = results_df.loc[best_idx, 'feature_list']
    best_cv_score = results_df.loc[best_idx, 'cv_score']
    
    print(f"\n{'='*60}")
    print(f"BEST FEATURE SET (Iteration {best_idx}):")
    print(f"Features: {best_features}")
    print(f"CV ROC-AUC: {best_cv_score:.4f}")
    print(f"{'='*60}\n")
    
    return best_features, results_df


# Run it again
selected_features, elimination_results = backward_elimination_optimized(
    X_train, y_train, best_C=best_C, cv=cv, metric='roc_auc'
)


# def backward_elimination_optimized(X, y, best_C, cv, metric='roc_auc'):
#     """
#     Backward elimination using the best hyperparameters found in GridSearch
#     """
#     current_features = list(X.columns)
#     results = []
    
#     while len(current_features) > 1:
#         # Train model with BEST C parameter
#         pipe = Pipeline([
#             ("type_cast", binary_to_cat),
#             ("preprocessor", preprocessor),
#             ("model", LogisticRegression(C=best_C, max_iter=1000, random_state=42))
#         ])
        
#         scores = cross_validate(
#             pipe, X[current_features], y, 
#             cv=cv, scoring=metric, return_train_score=True
#         )
        
#         mean_score = scores['test_score'].mean()
#         std_score = scores['test_score'].std()
        
#         results.append({
#             'iteration': len(results),
#             'n_features': len(current_features),
#             'cv_score': mean_score,
#             'cv_std': std_score,
#             'feature_list': current_features.copy()
#         })
        
#         print(f"Iteration {len(results):2d} | Features: {len(current_features):2d} | "
#               f"ROC-AUC: {mean_score:.4f} ± {std_score:.4f}")
        
#         # Train on full training set to get coefficients
#         pipe.fit(X[current_features], y)
#         coefficients = pipe.named_steps['model'].coef_[0]
        
#         # Find feature with smallest absolute coefficient
#         min_idx = np.argmin(np.abs(coefficients))
#         removed_feature = current_features[min_idx]
#         removed_coef = coefficients[min_idx]
        
#         print(f"           → Removing: '{removed_feature}' (coef: {removed_coef:.4f})\n")
#         current_features.pop(min_idx)
    
#     results_df = pd.DataFrame(results)
    
#     # Find best iteration (highest CV score)
#     best_idx = results_df['cv_score'].idxmax()
#     best_features = results_df.loc[best_idx, 'feature_list']
#     best_cv_score = results_df.loc[best_idx, 'cv_score']
    
#     print(f"\n{'='*60}")
#     print(f"BEST FEATURE SET (Iteration {best_idx}):")
#     print(f"Features: {best_features}")
#     print(f"CV ROC-AUC: {best_cv_score:.4f}")
#     print(f"{'='*60}\n")
    
#     return best_features, results_df


# # Run backward elimination with best C
# selected_features, elimination_results = backward_elimination_optimized(
#     X_train, y_train, best_C=best_C, cv=cv, metric='roc_auc'
# )

# ============================================================
# STEP 3: Train Final Model with Best Features + Best C
# ============================================================
print("="*60)
print("STEP 3: Final Model with Selected Features & Best C")
print("="*60)

final_pipe = Pipeline([
    ("type_cast", binary_to_cat),
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(C=best_C, max_iter=1000, random_state=42))
])

final_pipe.fit(X_train[selected_features], y_train)

# Evaluate on both metrics
train_preds = final_pipe.predict_proba(X_train[selected_features])[:,1]
test_preds = final_pipe.predict_proba(X_test[selected_features])[:,1]

print("\nTrain Metrics:")
print(f'  ROC-AUC: {roc_auc_score(y_train, train_preds):.4f}')
print(f'  PR-AUC:  {average_precision_score(y_train, train_preds):.4f}')

print("\nTest Metrics:")
print(f'  ROC-AUC: {roc_auc_score(y_test, test_preds):.4f}')
print(f'  PR-AUC:  {average_precision_score(y_test, test_preds):.4f}')

# Show feature importance
print(f"\nSelected {len(selected_features)} features out of {len(X_train.columns)}")
print("\nFinal Model Coefficients:")
coefficients = pd.DataFrame({
    'feature': selected_features,
    'coefficient': final_pipe.named_steps['model'].coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print(coefficients.to_string(index=False))

STEP 1: Finding Best Hyperparameters with GridSearch

✓ Best C: 1
✓ Best CV ROC-AUC: 0.9365

STEP 2: Backward Elimination with Best Parameters
Iteration  1 | Features:  8 | ROC-AUC: 0.9365 ± 0.0022
           → Removing: 'age' (coef: -0.4224)

Iteration  2 | Features:  7 | ROC-AUC: 0.9222 ± 0.0034
           → Removing: 'hypertension' (coef: -0.5949)



ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Daniel\Education\2025-09_AFI_Master_Data_Science\2026-05-31 Trabajo_En_Equipo\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Daniel\Education\2025-09_AFI_Master_Data_Science\2026-05-31 Trabajo_En_Equipo\.venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Daniel\Education\2025-09_AFI_Master_Data_Science\2026-05-31 Trabajo_En_Equipo\.venv\Lib\site-packages\sklearn\pipeline.py", line 613, in fit
    Xt = sel

In [ ]:
# log_pipe_line.fit(X_train, y_train)
# 
# feature_names = log_pipe_line.named_steps["preprocessor"].get_feature_names_out()
# 
# df_processed = log_pipe_line.transform(X_train)
# df_processed = pd.DataFrame(df_processed,columns=feature_names)
# df_processed.head()

#### Check the sense of the woe, if this is the same in optimal binning that in the woe_encoder.

Once that is ok we can adjust the logistic regresion.

In [ ]:
log_pipe_line

In [40]:
# Get WOE mapping
#woe_encoder = log_pipe_line.named_steps["preprocessor"].named_transformers_["woe"]
importlib.reload(up)  # reload utils to reflect recent changes
woe_encoder = (
    log_pipe_line
    .named_steps["preprocessor"]
    .named_transformers_["woe"]
    .named_steps["woe"]
)

woe_map = up.get_woe_mapping(woe_encoder)
woe_map

,feature,cat,ord_enc,woe_enc
0,gender,Male,1,-0.149763
1,gender,Female,2,0.118977
3,smoking_history,current,1,-0.223560
4,smoking_history,No Info,2,0.774097
5,smoking_history,never,3,-0.118364
6,smoking_history,former,4,-0.549811
8,hypertension,0,1,0.225959
9,hypertension,1,2,-1.438288
11,heart_disease,0,1,0.131395
12,heart_disease,1,2,-1.637085


In [41]:
# get binning and WOE mapping
bin_woe_encoder = log_pipe_line.named_steps["preprocessor"].named_transformers_["bin_woe"]
bin_woe_map = up.get_bin_woe_mapping(bin_woe_encoder)
bin_woe_map
#bin_woe_map[bin_woe_map['feature']=='blood_glucose_level']

,feature,Bin,WoE,IV
0,blood_glucose_level,"(-inf, 128.00)",1.612593,0.497041
1,blood_glucose_level,"[128.00, 159.50)",0.199551,0.016705
2,blood_glucose_level,"[159.50, 180.00)",-0.075264,0.000451
3,blood_glucose_level,"[180.00, inf)",-1.809115,0.698225
4,blood_glucose_level,Special,0.000000,0.000000
5,blood_glucose_level,Missing,0.000000,0.000000
6,HbA1c_level,"(-inf, 5.75)",1.809661,0.751757
7,HbA1c_level,"[5.75, 5.90)",0.121046,0.001170
8,HbA1c_level,"[5.90, 6.55)",0.094640,0.002844
9,HbA1c_level,"[6.55, inf)",-1.842499,0.838581
